In [ ]:
!pip install -q -U bitsandbytes

In [ ]:
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from diffusers import AutoPipelineForText2Image
import gradio as gr
import tempfile
import pandas as pd
import io

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU Available: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: CUDA is not available. Please switch Colab runtime to T4 GPU!")


PyTorch Version: 2.11.0+cu128
GPU Available: Tesla T4


In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

llm_model_id = "Qwen/Qwen2.5-7B-Instruct" # Alternatif: Qwen/Qwen2.5-7B-Instruct
tokenizer = AutoTokenizer.from_pretrained(llm_model_id)
llm_model = AutoModelForCausalLM.from_pretrained(llm_model_id, quantization_config=bnb_config, device_map="auto")
text_generator = pipeline("text-generation", model=llm_model, tokenizer=tokenizer)

print("Diffusion Modeli Yükleniyor...")
diffusion_model_id = "stabilityai/sdxl-turbo"
image_generator = AutoPipelineForText2Image.from_pretrained(
    diffusion_model_id,
    torch_dtype=torch.float16,
    variant="fp16"
).to("cuda")

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

In [ ]:
def generate_tabular(topic, num_rows):
    prompt = f"<|system|>\nSen veri bilimi için sentetik veri üreten bir asistansın. Sadece istenen CSV formatını döndür. Başka metin ekleme.</s>\n<|user|>\nKonu: {topic}. {num_rows} satırlık, virgülle ayrılmış (CSV) bir veri seti oluştur. En az 4 anlamlı sütun içersin.</s>\n<|assistant|>\n"
    result = text_generator(prompt, max_new_tokens=512, temperature=0.7)
    generated_text = result[0]['generated_text'].split("<|assistant|>\n")[-1].strip()

    # Veriyi bir Excel veya CSV dosyası olarak kaydetme
    try:
        # Metni okuyup .xlsx formatına çevirmeyi dener
        df = pd.read_csv(io.StringIO(generated_text), sep=",")
        temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".xlsx")
        df.to_excel(temp_file.name, index=False)
        file_path = temp_file.name
    except Exception:
        # Eğer okuma/dönüştürme başarısız olursa standart CSV olarak kaydeder
        temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".csv")
        with open(temp_file.name, "w", encoding="utf-8") as f:
            f.write(generated_text)
        file_path = temp_file.name

    return generated_text, file_path

def generate_qa(topic, num_pairs):
    prompt = f"<|system|>\nSen bir eğitim veri seti oluşturucususun. Sadece Soru-Cevap formatında çıktı ver.</s>\n<|user|>\n{topic} hakkında {num_pairs} adet detaylı Soru-Cevap çifti oluştur. \nFormat: \nSoru 1: ...\nCevap 1: ...</s>\n<|assistant|>\n"
    result = text_generator(prompt, max_new_tokens=1024, temperature=0.7)
    generated_text = result[0]['generated_text'].split("<|assistant|>\n")[-1].strip()

    # Metni .txt dosyası olarak kaydetme
    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".txt")
    with open(temp_file.name, "w", encoding="utf-8") as f:
        f.write(generated_text)

    return generated_text, temp_file.name

def generate_visual(prompt):
    image = image_generator(prompt=prompt, num_inference_steps=2, guidance_scale=0.0).images[0]

    # Görseli .png dosyası olarak kaydetme
    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".png")
    image.save(temp_file.name)

    return image, temp_file.name

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🧪 Sentetik Veri Üretim Fabrikası")
    gr.Markdown("LLM ve Diffusion modelleri ile farklı formatlarda eğitim verisi oluşturun ve anında indirin.")

    with gr.Tabs():
        # Tabular Veri Sekmesi
        with gr.Tab("📊 Tabular Veri (Excel/CSV)"):
            with gr.Row():
                with gr.Column():
                    tab_topic = gr.Textbox(label="Veri Seti Konusu", placeholder="Konu girin...")
                    tab_rows = gr.Slider(minimum=5, maximum=50, step=5, label="Satır Sayısı", value=10)
                    tab_btn = gr.Button("Veri Üret", variant="primary")
                with gr.Column():
                    tab_output = gr.TextArea(label="Üretilen Veri Önizlemesi", lines=10)
                    tab_download = gr.File(label="Veri Setini İndir (.xlsx / .csv)")
            tab_btn.click(fn=generate_tabular, inputs=[tab_topic, tab_rows], outputs=[tab_output, tab_download])

        # Soru-Cevap Veri Sekmesi
        with gr.Tab("💬 Soru-Cevap Verisi"):
            with gr.Row():
                with gr.Column():
                    qa_topic = gr.Textbox(label="Konu", placeholder="Konu girin...")
                    qa_pairs = gr.Slider(minimum=1, maximum=20, step=1, label="Soru Sayısı", value=5)
                    qa_btn = gr.Button("Veri Üret", variant="primary")
                with gr.Column():
                    qa_output = gr.TextArea(label="Üretilen Q&A Verisi", lines=10)
                    qa_download = gr.File(label="Veri Setini İndir (.txt)")
            qa_btn.click(fn=generate_qa, inputs=[qa_topic, qa_pairs], outputs=[qa_output, qa_download])

        # Görsel Veri Sekmesi
        with gr.Tab("🖼️ Görsel Veri"):
            with gr.Row():
                with gr.Column():
                    img_prompt = gr.Textbox(label="Görsel Promptu", placeholder="A highly detailed portrait...")
                    img_btn = gr.Button("Görsel Üret", variant="primary")
                with gr.Column():
                    img_output = gr.Image(label="Üretilen Görsel")
                    img_download = gr.File(label="Görseli İndir (.png)")
            img_btn.click(fn=generate_visual, inputs=[img_prompt], outputs=[img_output, img_download])

if __name__ == '__main__':
    demo.launch(debug=True, share=True)

/tmp/ipykernel_4420/4213136869.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://db9271d32a25507a5b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  

  0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/stable_diffusion_xl/pipeline_stable_diffusion_xl.py:748: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


...

  0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/stable_diffusion_xl/pipeline_stable_diffusion_xl.py:748: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


  0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/stable_diffusion_xl/pipeline_stable_diffusion_xl.py:748: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://db9271d32a25507a5b.gradio.live
